In [39]:
#import library
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [40]:
# 1. Download seluruh repository Sigaleh dari GitHub
!git clone https://github.com/ZuperGilz/Sigaleh.git

# 2. Arahkan path langsung ke folder dataset_mentah hasil clone
folder_path = '/content/Sigaleh/dataset_mentah'

# 3. Ambil semua path file Excel secara otomatis
file_list = glob.glob(os.path.join(folder_path, '*.xlsx'))

semua_data = []

print("Mulai menarik data dari GitHub dan memproses file...")

# 4. Looping ke setiap file
for file in file_list:
    df = pd.read_excel(file)

    # Pengecekan wilayah dari nama file
    nama_file = os.path.basename(file).lower()
    if 'padang' in nama_file:
        wilayah = 'Padang'
    elif 'bukittinggi' in nama_file:
        wilayah = 'Bukittinggi'
    else:
        wilayah = 'Lainnya'

    # Proses MELT (Wide to Long format)
    df_melted = pd.melt(
        df,
        id_vars=['No', 'Komoditas (Rp)'],
        var_name='Tanggal',
        value_name='Harga'
    )

    df_melted['Wilayah'] = wilayah
    semua_data.append(df_melted)

# 5. Gabungkan dan bersihkan data
if len(semua_data) > 0:
    df_final = pd.concat(semua_data, ignore_index=True)

    # Hapus kolom 'No'
    df_final = df_final.drop(columns=['No'])

    # Bersihkan angka harga dari koma dan ubah ke format numerik
    df_final['Harga'] = df_final['Harga'].astype(str).str.replace(',', '', regex=False)
    df_final['Harga'] = pd.to_numeric(df_final['Harga'], errors='coerce')

    # Bersihkan format Tanggal
    df_final['Tanggal'] = df_final['Tanggal'].astype(str).str.replace(' ', '')
    df_final['Tanggal'] = pd.to_datetime(df_final['Tanggal'], format='%d/%m/%Y', errors='coerce')

    # Urutkan secara kronologis berdasarkan Wilayah -> Komoditas -> Tanggal
    df_final = df_final.sort_values(by=['Wilayah', 'Komoditas (Rp)', 'Tanggal'], ascending=[True, True, True])
    df_final = df_final.reset_index(drop=True)

    print(f"\nBerhasil memproses {len(file_list)} file dari GitHub.")
    print(f"Bentuk data gabungan: {df_final.shape}")
    print("\nContoh 5 baris pertama:")
    print(df_final.head())

    # Export hasilnya
    df_final.to_csv('data_harga_komoditas.csv', index=False)
    print("\nSelesai! File 'data_harga_padi_long_format.csv' berhasil dibuat.")
else:
    print("Gagal menemukan file .xlsx. Cek kembali struktur folder di GitHub.")

fatal: destination path 'Sigaleh' already exists and is not an empty directory.
Mulai menarik data dari GitHub dan memproses file...

Berhasil memproses 12 file dari GitHub.
Bentuk data gabungan: (94830, 4)

Contoh 5 baris pertama:
  Komoditas (Rp)    Tanggal    Harga      Wilayah
0   Bawang Merah 2020-01-02  31000.0  Bukittinggi
1   Bawang Merah 2020-01-03  31000.0  Bukittinggi
2   Bawang Merah 2020-01-06  31000.0  Bukittinggi
3   Bawang Merah 2020-01-07  31500.0  Bukittinggi
4   Bawang Merah 2020-01-08  32250.0  Bukittinggi

Selesai! File 'data_harga_padi_long_format.csv' berhasil dibuat.


In [41]:
df_final.head()

,Komoditas (Rp),Tanggal,Harga,Wilayah
0,Bawang Merah,2020-01-02,31000.0,Bukittinggi
1,Bawang Merah,2020-01-03,31000.0,Bukittinggi
2,Bawang Merah,2020-01-06,31000.0,Bukittinggi
3,Bawang Merah,2020-01-07,31500.0,Bukittinggi
4,Bawang Merah,2020-01-08,32250.0,Bukittinggi


In [42]:
df_final.shape

(94830, 4)

In [48]:
# 1. Tentukan rentang waktu yang menutupi seluruh data harga kamu
# (Misalnya dari awal 2020 hingga April 2026)
start_date = "2020-01-01"
end_date = "2026-04-30"

print(f"Mengambil data cuaca historis dari {start_date} hingga {end_date}...")

# 2. Kordinat Geografis Padang dan Bukittinggi
kordinat = {
    'Padang': {'lat': -0.9471, 'lon': 100.3658},
    'Bukittinggi': {'lat': -0.3056, 'lon': 100.3692}
}

data_cuaca_list = []

# 3. Looping untuk mengambil data cuaca dari API
for kota, lokasi in kordinat.items():
    url = "https://archive-api.open-meteo.com/v1/archive"

    # Menambahkan variabel cuaca yang lebih detail untuk analisis pertanian
    params = {
        "latitude": lokasi['lat'],
        "longitude": lokasi['lon'],
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",        # Suhu rata-rata harian (°C)
            "precipitation_sum",          # Total curah hujan harian (mm)
            "shortwave_radiation_sum",    # Total radiasi matahari/intensitas cahaya (MJ/m²)
            "wind_speed_10m_max"          # Kecepatan angin maksimal (km/h)
            # Catatan: Open-Meteo daily archive tidak selalu punya 'relative_humidity_2m_mean' secara langsung,
            # biasanya harus ditarik per jam lalu dirata-rata. Tapi 4 variabel ini sudah sangat merepresentasikan iklim.
        ],
        "timezone": "Asia/Jakarta"
    }

    response = requests.get(url, params=params)

    # Cek apakah request berhasil
    if response.status_code == 200:
        data = response.json()

        # Ekstrak data JSON ke dalam DataFrame pandas
        df_kota = pd.DataFrame({
            'Tanggal': pd.to_datetime(data['daily']['time']),
            'Suhu_Rata2_C': data['daily']['temperature_2m_mean'],
            'Curah_Hujan_mm': data['daily']['precipitation_sum'],
            'Radiasi_Matahari_MJm2': data['daily']['shortwave_radiation_sum'],
            'Kecepatan_Angin_Max_kmh': data['daily']['wind_speed_10m_max']
        })

        df_kota['Wilayah'] = kota
        data_cuaca_list.append(df_kota)
        print(f"Berhasil menarik data cuaca untuk {kota}")
    else:
        print(f"Gagal mengambil data untuk {kota}. Status code: {response.status_code}")

# 4. Gabungkan dan simpan data cuaca
if data_cuaca_list:
    df_cuaca_final = pd.concat(data_cuaca_list, ignore_index=True)

    # Rapihkan urutan kolom
    df_cuaca_final = df_cuaca_final[['Wilayah', 'Tanggal', 'Suhu_Rata2_C', 'Curah_Hujan_mm', 'Radiasi_Matahari_MJm2', 'Kecepatan_Angin_Max_kmh']]

    # Export menjadi CSV tersendiri
    df_cuaca_final.to_csv('dataset_cuaca_padang_bukittinggi.csv', index=False)

    print("\nBentuk data cuaca:", df_cuaca_final.shape)
    print("\nContoh 5 baris pertama:")
    print(df_cuaca_final.head())
    print("\nSelesai! File 'dataset_cuaca_padang_bukittinggi.csv' berhasil dibuat secara mandiri.")

Mengambil data cuaca historis dari 2020-01-01 hingga 2026-04-30...
Berhasil menarik data cuaca untuk Padang
Berhasil menarik data cuaca untuk Bukittinggi

Bentuk data cuaca: (4624, 6)

Contoh 5 baris pertama:
  Wilayah    Tanggal  Suhu_Rata2_C  Curah_Hujan_mm  Radiasi_Matahari_MJm2  \
0  Padang 2020-01-01          27.3             9.5                  18.55   
1  Padang 2020-01-02          27.7             2.9                  21.86   
2  Padang 2020-01-03          26.6            13.9                  16.96   
3  Padang 2020-01-04          26.7             8.3                  19.58   
4  Padang 2020-01-05          27.0             7.6                  18.90   

   Kecepatan_Angin_Max_kmh  
0                      8.4  
1                      9.2  
2                      7.8  
3                      7.2  
4                      9.7  

Selesai! File 'dataset_cuaca_padang_bukittinggi.csv' berhasil dibuat secara mandiri.


In [50]:
df1 = pd.read_csv('data_harga_komoditas.csv')
df2 = pd.read_csv('dataset_cuaca_padang_bukittinggi.csv')

print(df1.shape)
print(df2.shape)

(94830, 4)
(4624, 6)
